# Economic Forecasting Model

This notebook implements an economic forecasting model using SageMaker. We'll:
1. Load and preprocess economic data
2. Train a Prophet model for time series forecasting
3. Implement LSTM model for complex patterns
4. Evaluate and compare model performance
5. Deploy the best model to production

## Data Sources
- Federal Reserve Economic Data (FRED)
- World Bank Open Data
- IMF Data

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_datareader.data as web
from datetime import datetime, timedelta
from prophet import Prophet
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import boto3
import sagemaker
from sagemaker.pytorch import PyTorch
from sagemaker import get_execution_role

# Initialize SageMaker session
role = get_execution_role()
session = sagemaker.Session()
bucket = session.default_bucket()

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

## Data Collection and Preprocessing

Let's fetch economic indicators from FRED and process them for our model.

In [ ]:
# Define economic indicators to track
indicators = {
    'GDP': 'GDP',           # Gross Domestic Product
    'UNRATE': 'UNRATE',     # Unemployment Rate
    'CPIAUCSL': 'CPIAUCSL', # Consumer Price Index
    'FEDFUNDS': 'FEDFUNDS', # Federal Funds Rate
    'M2': 'M2'              # Money Supply M2
}

# Fetch data from FRED
start_date = '2010-01-01'
end_date = datetime.now()

def fetch_economic_data():
    data = {}
    for name, symbol in indicators.items():
        try:
            df = web.DataReader(symbol, 'fred', start_date, end_date)
            data[name] = df
        except Exception as e:
            print(f"Error fetching {name}: {str(e)}")
    return data

economic_data = fetch_economic_data()

# Combine all indicators into one DataFrame
combined_data = pd.DataFrame()
for name, df in economic_data.items():
    combined_data[name] = df[df.columns[0]]

# Handle missing values
combined_data = combined_data.fillna(method='ffill').fillna(method='bfill')

# Create date features
combined_data['year'] = combined_data.index.year
combined_data['month'] = combined_data.index.month
combined_data['quarter'] = combined_data.index.quarter

print("Data shape:", combined_data.shape)
combined_data.head()

## Prophet Model Implementation

First, we'll implement Facebook's Prophet model for time series forecasting.

In [ ]:
# Prepare data for Prophet
def prepare_prophet_data(data, indicator):
    df = data[[indicator]].reset_index()
    df.columns = ['ds', 'y']
    return df

# Train Prophet model
def train_prophet_model(data, indicator, future_periods=12):
    # Prepare data
    prophet_data = prepare_prophet_data(data, indicator)
    
    # Initialize and train model
    model = Prophet(yearly_seasonality=True, 
                   weekly_seasonality=False,
                   daily_seasonality=False)
    model.fit(prophet_data)
    
    # Make future predictions
    future = model.make_future_dataframe(periods=future_periods, freq='M')
    forecast = model.predict(future)
    
    return model, forecast

# Train models for each indicator
prophet_models = {}
prophet_forecasts = {}

for indicator in indicators.keys():
    print(f"\nTraining Prophet model for {indicator}")
    model, forecast = train_prophet_model(combined_data, indicator)
    prophet_models[indicator] = model
    prophet_forecasts[indicator] = forecast

# Display forecasts
for indicator in indicators.keys():
    forecast = prophet_forecasts[indicator]
    print(f"\nForecast for {indicator}:")
    print(forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail())

## LSTM Model Implementation

Now, let's implement a more complex LSTM model for capturing non-linear patterns.

In [ ]:
# Define LSTM model
class EconomicLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(EconomicLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out

# Prepare data for LSTM
def prepare_lstm_data(data, lookback=12):
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(data)
    
    X, y = [], []
    for i in range(len(scaled_data) - lookback):
        X.append(scaled_data[i:(i + lookback)])
        y.append(scaled_data[i + lookback])
    
    return np.array(X), np.array(y), scaler

# Train LSTM model
def train_lstm_model(data, indicator, epochs=100):
    # Prepare data
    X, y, scaler = prepare_lstm_data(data[[indicator]])
    train_size = int(len(X) * 0.8)
    
    X_train = torch.FloatTensor(X[:train_size])
    y_train = torch.FloatTensor(y[:train_size])
    X_test = torch.FloatTensor(X[train_size:])
    y_test = torch.FloatTensor(y[train_size:])
    
    # Initialize model
    model = EconomicLSTM(input_size=1, hidden_size=50, num_layers=2, output_size=1)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters())
    
    # Training loop
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')
    
    return model, scaler

# Train LSTM models for each indicator
lstm_models = {}
lstm_scalers = {}

for indicator in indicators.keys():
    print(f"\nTraining LSTM model for {indicator}")
    model, scaler = train_lstm_model(combined_data, indicator)
    lstm_models[indicator] = model
    lstm_scalers[indicator] = scaler

## Model Deployment

Let's package and deploy our models to SageMaker endpoints.

In [ ]:
# Save models to S3
def save_models_to_s3():
    for indicator in indicators.keys():
        # Save Prophet model
        prophet_path = f'models/prophet_{indicator}'
        prophet_models[indicator].save(prophet_path)
        
        # Save LSTM model
        lstm_path = f'models/lstm_{indicator}.pth'
        torch.save(lstm_models[indicator].state_dict(), lstm_path)
        
        # Upload to S3
        session.upload_data(
            prophet_path,
            bucket=bucket,
            key_prefix=f'models/prophet_{indicator}'
        )
        session.upload_data(
            lstm_path,
            bucket=bucket,
            key_prefix=f'models/lstm_{indicator}'
        )

# Deploy models as SageMaker endpoints
def deploy_sagemaker_endpoint(model_data, model_name):
    pytorch_model = PyTorch(
        entry_point='predict.py',
        source_dir='code',
        role=role,
        framework_version='1.8.1',
        py_version='py36',
        model_data=model_data,
        instance_count=1,
        instance_type='ml.m5.xlarge'
    )
    
    predictor = pytorch_model.deploy(
        initial_instance_count=1,
        instance_type='ml.m5.xlarge',
        endpoint_name=f'econosense-{model_name}'
    )
    
    return predictor

# Save and deploy models
save_models_to_s3()
print("Models saved to S3 successfully!")